In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

In [ ]:
bronze_path = "tihim_project.bronze.customers"
silver_path = "tihim_project.silver.customers"
quarantine_path = "tihim_project.silver.customers_quarantine"
checkpoint_path = "/Volumes/tihim_project/ops/stream_state/checkpoints/silver/customers"


In [ ]:

spark.sql(f"""   
    CREATE TABLE IF NOT EXISTS {silver_path} (
        customer_id     STRING,
        full_name       STRING,
        email           STRING,
        domain          STRING,
        city            STRING,
        state           STRING,
        ingestion_date  TIMESTAMP,
        create_date     TIMESTAMP,
        update_date     TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (delta.enableChangeDataFeed = true) 
""")




In [ ]:
def cleaned_customers(df):
    return (df
            .drop("_rescued_data", "source_file")
            .withColumn("full_name", F.initcap(F.trim(F.concat_ws(" ", F.col("first_name"), F.col("last_name")))))
            .drop("first_name", "last_name")
            .withColumn("email", F.lower(F.trim(F.col("email"))))
            .withColumn("city",  F.initcap(F.trim(F.col("city"))))
            .withColumn("state", F.upper(F.trim(F.col("state")))))
    

In [ ]:
def flag_customers(df):
    regex = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"
    reason = F.concat_ws("; ", F.when(F.col("customer_id").isNull(), F.lit("null customer_id")),
                         F.when(F.col("full_name").isNull(), F.lit("null full_name")),
                         F.when(F.col("email").isNull(), F.lit("null email")),
                         F.when(F.col("email").isNotNull() & ~F.col("email").rlike(regex), F.lit("invalid email"))        
    )
    return df.withColumn("dq_reason", reason) 
            

In [ ]:
def upsert_customers_to_silver(microBatchDf, batchId):
    flagged = flag_customers(cleaned_customers(microBatchDf))

    valid = flagged.filter(F.col("dq_reason") == "")
    rejects = flagged.filter(F.col("dq_reason") != "")


    if not rejects.isEmpty():
        (
            rejects.withColumn("batch_id", F.lit(batchId))
                .withColumn("rejected_at", F.current_timestamp())
                .write.format("delta").mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(quarantine_path)
        )

    w = Window.partitionBy("customer_id").orderBy(F.col("ingestion_date").desc())
    final = (
        valid.withColumn("rn", F.row_number().over(w)).filter(F.col("rn")==1).drop("rn", "dq_reason")
        .withColumn("domain", F.split(F.col("email"), "@")[1])
        .withColumn("create_date", F.col("ingestion_date"))
        .withColumn("update_date", F.col("ingestion_date"))
    )
    
    try:
        (DeltaTable.forName(spark, silver_path).alias("target").merge(
            source = final.alias("update"),
            condition = "target.customer_id = update.customer_id"
            ).whenMatchedUpdate(

                condition = """
                    NOT (target.email <=> update.email) OR
                    NOT (target.city <=> update.city) OR
                    NOT (target.state <=> update.state) OR
                    NOT (target.domain <=> update.domain) OR
                    NOT (target.full_name <=> update.full_name) 
                """,
                set = {
                    "email" : "update.email",
                    "city" : "update.city",
                    "state" : "update.state",
                    "domain" : "update.domain",
                    "full_name" : "update.full_name",
                    "update_date" : "update.update_date"
                }

            ).whenNotMatchedInsert(
                values = {
                    "customer_id" : "update.customer_id",
                    "email" : "update.email",
                    "city" : "update.city",
                    "state" : "update.state",
                    "domain" : "update.domain",
                    "full_name" : "update.full_name",
                    "create_date" : "update.ingestion_date",
                    "update_date" : "update.ingestion_date",
                    "ingestion_date" : "update.ingestion_date"

                }
            ).execute()
        )
    except Exception as e:
        print(f"[silver_customers] batch {batchId} failed: {e}")
        raise
    
    print(f"[silver_customers] batch {batchId}: {final.count()} valid, {rejects.count()} quarantined")

In [ ]:
query = spark.readStream.table(bronze_path)\
        .writeStream\
        .foreachBatch(upsert_customers_to_silver)\
        .option("checkpointLocation", checkpoint_path)\
        .trigger(availableNow=True)\
        .start()

query.awaitTermination()